# Checkpoint Evaluation & Ablations

Evaluates existing checkpoints from Google Drive — no retraining needed.

| Section | What it does |
|---|---|
| 1 | Setup & pytest |
| 2 | List checkpoints + training curve |
| 3 | Greedy baseline eval (beam=1) |
| 4 | Eval sweep — beam width × length normalisation |
| 5 | Cross-epoch checkpoint comparison |
| 6 | Numerical visualisations (heatmap, bar chart, radar) |
| 7 | Attention map comparison across checkpoints |
| 8 | Caption comparison — greedy vs beam=5 on the same images |

**Before running:** `Runtime → Change runtime type → GPU`

## 1. Setup & Validate Code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
from pathlib import Path

# ── Edit if your folder is in a different location ───────────────────────────
REPO_DIR   = Path('/content/drive/MyDrive/Neural_Image_Caption_Generator')
CKPT_DIR   = REPO_DIR / 'checkpoints'
DATA_ROOT  = REPO_DIR / 'data' / 'flickr8k'
VOCAB_PATH = DATA_ROOT / 'vocab.json'
RESULTS_DIR = REPO_DIR / 'results' / 'ablation_experiments'
# ─────────────────────────────────────────────────────────────────────────────

assert REPO_DIR.exists(), f'Repo not found: {REPO_DIR}'
assert CKPT_DIR.exists(), f'Checkpoints not found: {CKPT_DIR}'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Working in:  ', Path.cwd())
print('Results dir: ', RESULTS_DIR)

In [ ]:
!python -m pip install -q -r requirements.txt
import torch
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
!pytest -q

## 2. List Checkpoints + Training Curve

In [ ]:
import torch, csv, matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

epoch_ckpts = sorted(CKPT_DIR.glob('epoch_*.pt'))
best_ckpt   = CKPT_DIR / 'best.pt'

# ── Read val BLEU-4 stored in each checkpoint ────────────────────────────────
epoch_nums, val_bleu4s = [], []
for ckpt in epoch_ckpts:
    d = torch.load(ckpt, map_location='cpu')
    ep = d.get('epoch')
    b4 = d.get('val_bleu4', d.get('val_scores', {}).get('bleu4'))
    if ep and b4:
        epoch_nums.append(ep)
        val_bleu4s.append(b4 * 100)

# ── Print table ──────────────────────────────────────────────────────────────
print(f'Found {len(epoch_ckpts)} epoch checkpoints + best.pt\n')
print(f'{"File":<22}  {"Size MB":>8}  {"val BLEU-4":>10}')
print('-' * 46)
for ckpt in epoch_ckpts:
    mb = ckpt.stat().st_size / 1e6
    ep = int(ckpt.stem.split('_')[1])
    b4_str = f'{val_bleu4s[epoch_nums.index(ep)]:.2f}' if ep in epoch_nums else '?'
    print(f'{ckpt.name:<22}  {mb:>8.1f}  {b4_str:>10}')
if best_ckpt.exists():
    d = torch.load(best_ckpt, map_location='cpu')
    b4 = d.get('val_bleu4', float('nan'))
    mb = best_ckpt.stat().st_size / 1e6
    print(f'{"best.pt":<22}  {mb:>8.1f}  {b4*100:>10.2f}  ← epoch {d.get("epoch","?")}')

In [ ]:
train_losses = {}
log_path = REPO_DIR / 'results' / 'training_log.csv'   # written by train.py, not an ablation output
if log_path.exists():
    with open(log_path) as f:
        for row in csv.DictReader(f):
            train_losses[int(row['epoch'])] = float(row['train_loss'])

ncols = 2 if train_losses else 1
fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 4))
if ncols == 1: axes = [axes]

best_ep = epoch_nums[int(np.argmax(val_bleu4s))] if val_bleu4s else None
ax = axes[0]
ax.plot(epoch_nums, val_bleu4s, 'o-', color='steelblue', linewidth=2)
ax.axhline(19.5, color='red', linestyle='--', alpha=0.6, label='Paper target (19.5)')
if best_ep:
    ax.axvline(best_ep, color='green', linestyle=':', alpha=0.8, label=f'best.pt (ep {best_ep})')
ax.axvline(6, color='orange', linestyle=':', alpha=0.7, label='Encoder fine-tune start')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val BLEU-4'); ax.set_title('Validation BLEU-4')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

if train_losses:
    eps = sorted(train_losses)
    axes[1].plot(eps, [train_losses[e] for e in eps], 'o-', color='darkorange', linewidth=2)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Train Loss')
    axes[1].set_title('Training Loss'); axes[1].grid(alpha=0.3)

plt.tight_layout()
out = RESULTS_DIR / 'training_curve.png'
plt.savefig(str(out), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')
if val_bleu4s:
    print(f'Peak val BLEU-4: {max(val_bleu4s):.2f} at epoch {best_ep}')

## 3. Greedy Baseline (beam=1) — best.pt

In [ ]:
!python evaluate.py \
  --checkpoint "{CKPT_DIR}/best.pt" \
  --data_root  "{DATA_ROOT}" \
  --vocab      "{VOCAB_PATH}" \
  --split test --beam_width 1 --batch_size 64 \
  --results_out "{RESULTS_DIR}/eval_greedy_baseline.json"

## 4. Eval Sweep — Beam Width × Length Normalisation
Tests all 5 decoding configurations against `best.pt`.

In [ ]:
import json, subprocess

SWEEP_CONFIGS = [
    dict(beam_width=1, length_normalize=False),
    dict(beam_width=3, length_normalize=False),
    dict(beam_width=5, length_normalize=False),
    dict(beam_width=3, length_normalize=True),
    dict(beam_width=5, length_normalize=True),
]

sweep_results = []
for cfg in SWEEP_CONFIGS:
    tag = f"beam{cfg['beam_width']}_ln{cfg['length_normalize']}"
    out = str(RESULTS_DIR / f'eval_{tag}.json')
    cmd = [
        'python', 'evaluate.py',
        '--checkpoint', str(CKPT_DIR / 'best.pt'),
        '--data_root',  str(DATA_ROOT),
        '--vocab',      str(VOCAB_PATH),
        '--split', 'test', '--batch_size', '64',
        '--beam_width', str(cfg['beam_width']),
        '--results_out', out,
    ]
    if cfg['length_normalize']: cmd.append('--length_normalize')
    subprocess.run(cmd, check=True)
    with open(out) as f: data = json.load(f)
    sweep_results.append({'label': tag.replace('_', ' '), **cfg, **data['scores']})
    print(f'  {tag}: BLEU-4={data["scores"]["bleu4"]*100:.2f}  METEOR={data["scores"].get("meteor",0)*100:.2f}')

## 5. Cross-Epoch Checkpoint Comparison
Evaluates key epoch checkpoints with beam=5 + length_normalize to show training progression.
Includes epoch 4 (pre-fine-tuning) vs later epochs (post-fine-tuning).

In [ ]:
all_epoch_ckpts = sorted(CKPT_DIR.glob('epoch_*.pt'))
last_ep = int(all_epoch_ckpts[-1].stem.split('_')[1]) if all_epoch_ckpts else 0

KEY_EPOCHS = []
for target in [1, 4, max(6, last_ep // 2), last_ep]:
    p = CKPT_DIR / f'epoch_{target:03d}.pt'
    if p.exists() and p not in KEY_EPOCHS:
        KEY_EPOCHS.append(p)
KEY_EPOCHS.append(CKPT_DIR / 'best.pt')

epoch_results = []
for ckpt in KEY_EPOCHS:
    out = str(RESULTS_DIR / f'epoch_eval_{ckpt.stem}.json')
    cmd = [
        'python', 'evaluate.py',
        '--checkpoint', str(ckpt),
        '--data_root',  str(DATA_ROOT),
        '--vocab',      str(VOCAB_PATH),
        '--split', 'test', '--batch_size', '64',
        '--beam_width', '5', '--length_normalize',
        '--results_out', out,
    ]
    subprocess.run(cmd, check=True)
    with open(out) as f: data = json.load(f)
    epoch_results.append({'label': ckpt.stem, **data['scores']})
    print(f'  {ckpt.stem}: BLEU-4={data["scores"]["bleu4"]*100:.2f}')

## 6. Numerical Visualisations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

METRICS  = ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'METEOR']
KEYS     = ['bleu1',  'bleu2',  'bleu3',  'bleu4',  'meteor']
PAPER    = [67.0,      44.8,     29.9,     19.5,     18.93  ]

sweep_labels = [r['label'] for r in sweep_results]
sweep_matrix = np.array([[r.get(k, 0)*100 for k in KEYS] for r in sweep_results])

fig, ax = plt.subplots(figsize=(9, 3.5))
im = ax.imshow(sweep_matrix, cmap='YlGn', aspect='auto',
               vmin=sweep_matrix.min() * 0.9, vmax=sweep_matrix.max() * 1.05)
ax.set_xticks(range(len(METRICS))); ax.set_xticklabels(METRICS, fontsize=10)
ax.set_yticks(range(len(sweep_labels))); ax.set_yticklabels(sweep_labels, fontsize=9)
ax.set_title('Eval Sweep: Beam × Length-Norm (best.pt)', fontsize=11, pad=10)
for i in range(len(sweep_labels)):
    for j in range(len(METRICS)):
        val = sweep_matrix[i, j]
        diff = val - PAPER[j]
        color = 'white' if val > sweep_matrix[:, j].mean() else 'black'
        ax.text(j, i, f'{val:.1f}\n({diff:+.1f})', ha='center', va='center',
                fontsize=7.5, color=color)
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
plt.tight_layout()
out = RESULTS_DIR / 'viz_heatmap_sweep.png'
plt.savefig(str(out), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

In [ ]:
n_configs = len(sweep_results)
x = np.arange(len(METRICS))
width = 0.75 / (n_configs + 1)
COLORS = plt.cm.tab10(np.linspace(0, 0.8, n_configs))

fig, ax = plt.subplots(figsize=(13, 5))
for i, (r, color) in enumerate(zip(sweep_results, COLORS)):
    vals = [r.get(k, 0)*100 for k in KEYS]
    ax.bar(x + i*width - (n_configs*width)/2 + width/2, vals, width,
           label=r['label'], color=color, alpha=0.85)
ax.plot(x, PAPER, 'rv-', markersize=8, linewidth=2,
        label='Paper Soft-Att (Table 1)', zorder=5)
ax.set_xticks(x); ax.set_xticklabels(METRICS, fontsize=11)
ax.set_ylabel('Score'); ax.set_title('Beam × Length-Norm Ablation vs Paper Baseline')
ax.legend(fontsize=8, loc='upper right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = RESULTS_DIR / 'viz_bar_sweep.png'
plt.savefig(str(out), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

In [ ]:
n_ep = len(epoch_results)
x = np.arange(len(METRICS))
width = 0.75 / (n_ep + 1)
COLORS2 = plt.cm.viridis(np.linspace(0.2, 0.9, n_ep))

fig, ax = plt.subplots(figsize=(13, 5))
for i, (r, color) in enumerate(zip(epoch_results, COLORS2)):
    vals = [r.get(k, 0)*100 for k in KEYS]
    ax.bar(x + i*width - (n_ep*width)/2 + width/2, vals, width,
           label=r['label'], color=color, alpha=0.85)
ax.plot(x, PAPER, 'rv-', markersize=8, linewidth=2,
        label='Paper Soft-Att (Table 1)', zorder=5)
ax.set_xticks(x); ax.set_xticklabels(METRICS, fontsize=11)
ax.set_ylabel('Score'); ax.set_title('Checkpoint Progression (beam=5, len_norm=True) vs Paper')
ax.legend(fontsize=8, loc='upper right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = RESULTS_DIR / 'viz_bar_epochs.png'
plt.savefig(str(out), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

In [ ]:
N = len(METRICS)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

def to_radar(row):
    vals = [row.get(k, 0)*100 / p for k, p in zip(KEYS, PAPER)]
    return vals + [vals[0]]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
paper_norm = [1.0] * N + [1.0]
ax.plot(angles, paper_norm, 'r--', linewidth=2, label='Paper (1.0 = paper score)')
ax.fill(angles, paper_norm, alpha=0.05, color='red')

COLORS3 = plt.cm.tab10(np.linspace(0, 0.8, len(sweep_results)))
for r, color in zip(sweep_results, COLORS3):
    vals = to_radar(r)
    ax.plot(angles, vals, 'o-', linewidth=1.5, color=color, label=r['label'])
    ax.fill(angles, vals, alpha=0.07, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(METRICS, fontsize=11)
ax.set_ylim(0, 1.5)
ax.set_yticks([0.5, 1.0, 1.25])
ax.set_yticklabels(['0.5×', '1.0× (paper)', '1.25×'], fontsize=8)
ax.set_title('Metrics Relative to Paper Baseline', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.tight_layout()
out = RESULTS_DIR / 'viz_radar.png'
plt.savefig(str(out), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

In [ ]:
# ── (E) Summary table printed to screen ─────────────────────────────────────
def print_results_table(rows, title):
    print(f'\n{title}')
    print('=' * 80)
    print(f'{"Config":<28}  {"BLEU-1":>7}  {"BLEU-2":>7}  {"BLEU-3":>7}  {"BLEU-4":>7}  {"METEOR":>7}')
    print('-' * 80)
    print(f'{"Paper Soft-Att (Table 1)":<28}  {67.0:>7.1f}  {44.8:>7.1f}  {29.9:>7.1f}  {19.5:>7.1f}  {18.93:>7.2f}')
    print('-' * 80)
    for r in rows:
        m = f"{r['meteor']*100:>7.2f}" if 'meteor' in r else '    N/A'
        print(f"{r['label']:<28}  {r['bleu1']*100:>7.2f}  {r['bleu2']*100:>7.2f}  "
              f"{r['bleu3']*100:>7.2f}  {r['bleu4']*100:>7.2f}  {m}")
    print('=' * 80)

print_results_table(sweep_results,  'BEAM × LENGTH-NORM SWEEP  (best.pt)')
print_results_table(epoch_results,  'CHECKPOINT PROGRESSION  (beam=5, len_norm)')

## 7. Attention Map Comparison Across Checkpoints

Loads the same test images through multiple checkpoints and renders side-by-side
attention grids — one row per checkpoint, one column per generated word.
Shows how attention sharpens/shifts as training progresses.

In [ ]:
import math, torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from torchvision import transforms
from PIL import Image as PILImage

from config import ATTENTION_DIM, DECODER_DIM, EMBED_DIM, MAX_DECODE_LEN
from models import Encoder, Decoder
from utils import Vocabulary, greedy_decode

_MEAN = np.array([0.485, 0.456, 0.406])
_STD  = np.array([0.229, 0.224, 0.225])
_TRANSFORM = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN.tolist(), _STD.tolist()),
])

def unnorm(t):
    img = t.permute(1,2,0).numpy() * _STD + _MEAN
    return (np.clip(img, 0, 1) * 255).astype(np.uint8)

@torch.no_grad()
def load_model_from_ckpt(ckpt_path, vocab, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    enc = Encoder(fine_tune=False).to(device)
    dec = Decoder(
        attention_dim=ATTENTION_DIM, embed_dim=EMBED_DIM,
        decoder_dim=DECODER_DIM, vocab_size=len(vocab), dropout=0.0,
    ).to(device)
    enc.load_state_dict(ckpt['encoder'])
    dec.load_state_dict(ckpt['decoder'])
    enc.eval(); dec.eval()
    return enc, dec

@torch.no_grad()
def decode_with_attention(enc, dec, img_tensor, vocab, device):
    caption, alphas, token_ids = greedy_decode(
        enc, dec, img_tensor.unsqueeze(0).to(device), vocab, device,
        max_len=MAX_DECODE_LEN,
    )
    words = [vocab.idx2word.get(i, '<unk>') for i in token_ids]
    return caption, words, alphas   # alphas: (T, L)

def overlay_alpha(img_np, alpha_1d, sigma=8.0):
    """Return (224,224,3) paper-style grayscale+white-highlight overlay."""
    L = len(alpha_1d)
    grid = int(round(L**0.5))
    alpha_map = torch.tensor(alpha_1d).view(1,1,grid,grid)
    up = F.interpolate(alpha_map, (224,224), mode='bilinear', align_corners=False).squeeze().numpy()
    up = gaussian_filter(up, sigma=sigma)
    if up.max() > 0: up = up / up.max()
    gray = np.dot(img_np/255.0, [0.299, 0.587, 0.114])
    rgb = np.stack([gray]*3, axis=-1)
    rgb = rgb + up[:,:,None] * 0.7   # brighten attended regions
    return (np.clip(rgb, 0, 1) * 255).astype(np.uint8)

print('Helpers loaded.')

In [ ]:
from utils import load_flickr8k_captions

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab  = Vocabulary.load(str(VOCAB_PATH))

# Pick N test images to visualise
N_IMAGES = 4
img_to_caps = load_flickr8k_captions(str(DATA_ROOT), 'test')
sample_names = sorted(img_to_caps.keys())[:N_IMAGES]
sample_paths = [str(DATA_ROOT / 'images' / n) for n in sample_names]
sample_tensors = [_TRANSFORM(PILImage.open(p).convert('RGB')) for p in sample_paths]

# Checkpoints to compare (auto-selects; adjust as needed)
COMPARE_CKPTS = {
    f'epoch_{1:03d} (start)':         CKPT_DIR / f'epoch_{1:03d}.pt',
    f'epoch_{4:03d} (pre-finetune)':   CKPT_DIR / f'epoch_{4:03d}.pt',
    'best.pt':                          CKPT_DIR / 'best.pt',
}
# Remove entries whose files don't exist
COMPARE_CKPTS = {k: v for k, v in COMPARE_CKPTS.items() if v.exists()}
print('Comparing checkpoints:', list(COMPARE_CKPTS.keys()))

In [ ]:
MAX_WORDS = 8

for img_idx, (img_tensor, img_path) in enumerate(zip(sample_tensors, sample_paths)):
    img_np = unnorm(img_tensor)
    n_rows = len(COMPARE_CKPTS)
    n_cols = MAX_WORDS + 1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2 * n_cols, 2.8 * n_rows))
    if n_rows == 1: axes = axes[np.newaxis, :]

    for row_idx, (ckpt_label, ckpt_path) in enumerate(COMPARE_CKPTS.items()):
        enc, dec = load_model_from_ckpt(str(ckpt_path), vocab, device)
        caption, words, alphas = decode_with_attention(enc, dec, img_tensor, vocab, device)

        axes[row_idx, 0].imshow(img_np)
        axes[row_idx, 0].set_title(ckpt_label, fontsize=7, fontweight='bold')
        axes[row_idx, 0].set_xlabel(f'"{caption[:50]}"', fontsize=6, labelpad=2)
        axes[row_idx, 0].axis('off')

        for w_idx in range(MAX_WORDS):
            ax = axes[row_idx, w_idx + 1]
            if w_idx < len(words) and w_idx < alphas.shape[0]:
                overlay = overlay_alpha(img_np, alphas[w_idx].numpy())
                ax.imshow(overlay)
                ax.set_title(words[w_idx], fontsize=8)
            ax.axis('off')

        del enc, dec
        torch.cuda.empty_cache()

    img_name = Path(img_path).stem
    fig.suptitle(f'Attention Comparison — {img_name}', fontsize=11, y=1.01)
    plt.tight_layout()
    out = RESULTS_DIR / f'attn_compare_{img_idx+1:02d}.png'
    plt.savefig(str(out), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')

## 8. Caption Comparison — Same Images, Different Decoding

For each test image: shows the image and the captions generated by
greedy (beam=1), beam=3, and beam=5 with length normalisation.
Also shows the gold reference captions.

In [ ]:
from utils import beam_search_decode

enc_best, dec_best = load_model_from_ckpt(str(CKPT_DIR / 'best.pt'), vocab, device)

DECODE_CONFIGS = [
    ('Greedy (beam=1)',   dict(beam_width=1, length_normalize=False)),
    ('Beam=3',            dict(beam_width=3, length_normalize=False)),
    ('Beam=5',            dict(beam_width=5, length_normalize=False)),
    ('Beam=5 + len-norm', dict(beam_width=5, length_normalize=True)),
]

@torch.no_grad()
def get_captions(enc, dec, img_tensor, vocab, device, configs):
    results = {}
    img_batch = img_tensor.unsqueeze(0).to(device)
    for label, cfg in configs:
        bw, ln = cfg['beam_width'], cfg['length_normalize']
        if bw == 1:
            cap, _, _ = greedy_decode(enc, dec, img_batch, vocab, device, max_len=MAX_DECODE_LEN)
        else:
            cap = beam_search_decode(enc, dec, img_batch, vocab, device,
                                     beam_width=bw, max_len=MAX_DECODE_LEN, length_normalize=ln)
        results[label] = cap
    return results

fig, axes = plt.subplots(1, N_IMAGES, figsize=(4.5 * N_IMAGES, 7))
if N_IMAGES == 1: axes = [axes]

for ax, img_tensor, img_path, img_name in zip(axes, sample_tensors, sample_paths, sample_names):
    img_np = unnorm(img_tensor)
    captions = get_captions(enc_best, dec_best, img_tensor, vocab, device, DECODE_CONFIGS)
    refs = img_to_caps[img_name][:3]

    ax.imshow(img_np); ax.axis('off')
    text_lines = []
    for label, cap in captions.items():
        text_lines.append(f'[{label}]\n{cap}')
    text_lines += ['', '[References]'] + [f'• {r[:60]}' for r in refs]
    ax.set_xlabel('\n'.join(text_lines), fontsize=6.5, ha='left', x=0,
                  fontfamily='monospace', labelpad=6)

fig.suptitle('Caption Comparison: Greedy vs Beam Search', fontsize=13, y=1.01)
plt.tight_layout()
out = RESULTS_DIR / 'caption_compare.png'
plt.savefig(str(out), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

del enc_best, dec_best
torch.cuda.empty_cache()

## Summary of Saved Outputs

In [ ]:
print(f'All outputs saved to: {RESULTS_DIR}\n')
print(f'{"File":<50}  {"Size KB":>10}')
print('-' * 64)
for f in sorted(RESULTS_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name:<48}  {f.stat().st_size/1e3:>10.1f}')